# Hito 3 ? R?plica y formalizaci?n

Ejecuta el protocolo corregido sobre EPM, CHEC, SSPD ZNI, XM y la fuente original vfjy-6fka. Escribe solo salidas dentro de repositorio/. Las dos capturas reales ya est?n conservadas; no necesita red. Consulte Docs/VERIFICACION_HITO3.md para l?mites y pendientes externos.


In [1]:
from pathlib import Path
import csv, json, subprocess, sys
root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "project.json").is_file()), None)
if root is None and (Path.cwd()/"repositorio/project.json").is_file():
    root = Path.cwd()/"repositorio"
if root is None:
    raise RuntimeError("Abra el notebook desde repositorio/ o desde la ra?z del proyecto")
print("Ra?z:", root.name)


Ra?z: repositorio


## Ejecutar las cinco fuentes sin cambios manuales

No se ejecutan cartas ni ML sobre vfjy-6fka: solo tiene ocho per?odos. No se interpola ni se fabrican meses. Los par?metros del algoritmo est?n fijados en Code/hito3/core.py y run.py.


In [2]:
result = subprocess.run([sys.executable, "-X", "utf8", "-B", "pipeline.py", "ejecutar", "hito3"], cwd=root, capture_output=True, text=True, encoding="utf-8")
print(result.stdout)
if result.returncode:
    raise RuntimeError(result.stderr)


Manifiesto: Data\gold\outputs\ejecuciones\manifiesto_20261114T011132384706Z.json
Ejecutando -m Code.hito3.run
fuentes=4 archivo=C:\Users\segep\OneDrive\Documents\Personal\Maestria en Ciencia de Datos\Semestre 3\Seminario III\Trabajo_final\Framework_calidad_datos_energia\repositorio\Data\gold\outputs\cartas\revision_cartas_fuentes.csv
Medición 2026-11-14T01:12:06Z

  control                      EPM                    CHEC                    SSPD                      XM
  R01        99.9724  medido        100.0000  medido        100.0000  medido        100.0000  medido      
  R02       100.0000  medido        100.0000  medido        100.0000  medido        100.0000  medido      
  R03       100.0000  provisional   100.0000  provisional   100.0000  medido        100.0000  medido      
  R04        99.7240  medido        100.0000  medido        100.0000  medido        100.0000  medido      
  R05       100.0000  medido         88.4385  provisional    91.6311  provisional   100.0000  medi

## Perfil, cartas y categor?as completas


In [3]:
out = root/"Data/gold/outputs/hito3"
with (out/"bootstrap_categorias.csv").open(encoding="utf-8", newline="") as stream:
    for row in csv.DictReader(stream):
        print(row["fuente"], "series:", row["series"], "categor?as:", row["categorias"], "estado:", row["estado"])


CHEC series: 272 categor?as: 138 estado: bootstrap_categorias
EPM series: 61 categor?as: 22 estado: bootstrap_categorias
SSPD series: 387 categor?as: 157 estado: bootstrap_categorias
SSPD_PLAN series: 0 categor?as: 0 estado: sin_observaciones
XM series: 1 categor?as: 1 estado: sin_IC_una_categoria


## Comprobar dos capturas independientes

Cada captura conserva :id, fecha UTC, p?ginas originales y SHA-256. Igualdad en dos cortes no garantiza estabilidad futura.


In [4]:
comparison = json.loads((out/"comparacion_descargas.json").read_text(encoding="utf-8"))
for row in comparison["comparaciones"]:
    assert row["mismos_registros"] and row["mismos_hallazgos_R01_R11"] and row["mismas_cartas_y_claves"]
    print(row["fuente"], "registros, hallazgos y cartas coinciden")


EPM registros, hallazgos y cartas coinciden
CHEC registros, hallazgos y cartas coinciden
SSPD_PLAN registros, hallazgos y cartas coinciden


## Verificar integridad y l?mites de la conclusi?n


In [5]:
check = subprocess.run([sys.executable, "-X", "utf8", "-B", "-m", "Code.hito3.verify"], cwd=root, capture_output=True, text=True, encoding="utf-8")
print(check.stdout)
if check.returncode:
    raise RuntimeError(check.stderr)
state = json.loads((out/"criterios_cierre.json").read_text(encoding="utf-8"))
print("Cierre:", state["estado"])
for pending in state["faltantes_externos"]:
    print("Pendiente:", pending)


{
  "estado": "correcto",
  "series_evaluadas": 742,
  "modelos_if": 26,
  "pares_descargas_verificados": 3,
  "cierre_integral": false,
  "nota": "Evidencia externa y aceptación académica pendientes; ver criterios_cierre.json"
}

Cierre: parcial_no_cierre_integral
Pendiente: Oportunidad real con plazo y disponibilidad verificables EPM/CHEC
Pendiente: Catálogos externos y aprobación de semántica/umbrales/responsables
Pendiente: Integración y aprobación del documento académico oficial
